# **Konvensi Umum dan Elemen API pada scikit-learn**
Catatan ini merangkum fondasi arsitektur perangkat lunak dari pustaka scikit-learn. Pembahasan difokuskan pada pemahaman konsep dasar Antarmuka Pemrograman Aplikasi (API), yang mencakup desain estimator, transformer, pipeline, optimasi hiperparameter, pemanfaatan metadata, serta standar praktik operasional terbaik (best practices).

#**Bagian 1: Filosofi Desain scikit-learn**
Pustaka scikit-learn direkayasa berdasarkan empat prinsip utama. Hampir seluruh algoritma machine learning di dalam ekosistem ini (seperti Regresi Linier, Random Forest, atau SVM) dirancang untuk mematuhi pola antarmuka yang identik, sehingga meminimalisasi kurva pembelajaran bagi praktisi data.

Secara fundamental, siklus hidup pemodelan dalam pustaka ini mengikuti alur:

Instansiasi objek model.

1.  Proses pelatihan algoritma menggunakan metode fit().

2. Menghasilkan luaran menggunakan predict() atau memodifikasi data menggunakan transform().

3. Mengevaluasi performa menggunakan metode score().





# **Bagian 2: Anatomi Estimator**
Estimator adalah representasi objek komputasi yang mempelajari pola dari data masukan. Hampir seluruh algoritma machine learning berstatus sebagai estimator. Objek ini mengandalkan dua metode eksekusi utama: fit() untuk fase pembelajaran data historis, dan predict() untuk fase inferensi atau peramalan data baru.

In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression

# 1. Definisi Data Simulasi
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

# 2. Instansiasi dan Pelatihan Model
model = LinearRegression()
model.fit(X, y)

# 3. Fase Inferensi (Prediksi)
prediksi = model.predict([[6], [7]])
print("Hasil Prediksi:", prediksi)

Hasil Prediksi: [5.75 6.7 ]


## **Kombinasi Fit dan Predict**
Pada paradigma Unsupervised Learning (pembelajaran tanpa pengawasan), metrik evaluasi sering kali berjalan serentak dengan klasifikasi. Algoritma seperti K-Means Clustering menyediakan metode pintas fit_predict() yang melatih model sekaligus mengembalikan label klaster dalam satu komputasi efisien.

In [2]:
from sklearn.cluster import KMeans

# Eksekusi gabungan pada algoritma K-Means
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)

print("Label Klaster:", labels)

Label Klaster: [0 0 0 1 1]


# **Bagian 3: Anatomi Transformer**
Transformer adalah kelas objek yang didedikasikan untuk memodifikasi atau merekayasa data mentah sebelum disalurkan ke dalam estimator prediktif. Operasi ini mencakup penskalaan (scaling), normalisasi, dan penyandian fitur (encoding).

In [3]:
from sklearn.preprocessing import StandardScaler

X_mentah = np.array([[1, 2], [3, 4], [5, 6]])

# Instansiasi Transformer
scaler = StandardScaler()

# Pendekatan Eksekusi Serentak
X_terskala = scaler.fit_transform(X_mentah)
print("Matriks Data Terskalakan:\n", X_terskala)

Matriks Data Terskalakan:
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


# **Bagian 4: Arsitektur Pipeline dan Otomatisasi Alur Kerja**
Pipeline adalah struktur data yang mengikat berbagai tahapan transformer dengan satu estimator final secara linear. Integrasi ini merupakan pilar dari praktik MLOps (Machine Learning Operations) modern karena mencegah insiden kebocoran data (data leakage) dan memastikan reproduksibilitas eksperimen.

In [4]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Persiapan Data
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, random_state=42)

# Konstruksi Pipeline Terintegrasi
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

# Eksekusi pelatihan pada seluruh arsitektur
pipe.fit(X_train, y_train)
print(f"Akurasi Pipeline Terintegrasi: {pipe.score(X_test, y_test):.4f}")

Akurasi Pipeline Terintegrasi: 0.9790


# **Bagian 5: Rekayasa Komponen Kustom (Custom Estimators)**
Fleksibilitas utama scikit-learn terletak pada kemudahan perancangan fungsi transformasi modifikasi mandiri. Dengan mewarisi atribut dari BaseEstimator dan TransformerMixin, komponen kustom yang dibuat akan sepenuhnya kompatibel dengan struktur Pipeline maupun mekanisme Cross-Validation.

In [5]:
from sklearn.base import BaseEstimator, TransformerMixin

# Pendefinisian Kelas Transformer Mandiri
class AddOneTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self # Tidak ada parameter internal yang perlu dipelajari

    def transform(self, X):
        return X + 1 # Operasi modifikasi matriks

# Simulasi Penggunaan
X_awal = np.array([[1], [2], [3]])
custom_transformer = AddOneTransformer()

print("Luaran Transformer Kustom:\n", custom_transformer.fit_transform(X_awal))

Luaran Transformer Kustom:
 [[2]
 [3]
 [4]]


# **Bagian 6: Atribut Internal dan Optimasi Hiperparameter (Tuning)**

Setelah model dilatih, parameter matematis yang dihasilkan (learned parameters) dapat diekstraksi melalui atribut berakhiran garis bawah (seperti coef_ untuk koefisien beban dan intercept_ untuk titik potong).

Di sisi lain, parameter yang diatur secara manual sebelum pelatihan disebut Hiperparameter. Untuk menemukan kombinasi konfigurasi yang memberikan performa maksimum, praktisi menggunakan algoritma pencarian sistematis seperti GridSearchCV.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import load_iris

iris = load_iris()

# 1. Definisi Ruang Pencarian (Search Space)
param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [3, 5, None]
}

# 2. Inisialisasi dan Eksekusi Grid Search dengan Cross-Validation
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3
)
grid.fit(iris.data, iris.target)

print("Konfigurasi Parameter Optimal:", grid.best_params_)
print(f"Skor Validasi Maksimum: {grid.best_score_:.4f}")

Konfigurasi Parameter Optimal: {'max_depth': 3, 'n_estimators': 50}
Skor Validasi Maksimum: 0.9667


# **Kesimpulan & Standar Praktik Terbaik (Best Practices)**
Penerapan kaidah arsitektur yang seragam adalah alasan utama mengapa scikit-learn menjadi fondasi operasional dalam industri sains data.